# NORTHSTAR -- Tower 20: Linux QA for Solace Browser

**Tower:** 20 -- Tower of Linux
**DNA:** `linux(native) = appimage(portable) x wayland(modern) x x11(compatible) x distros(ubuntu+fedora+arch) x flatpak(sandboxed) x cli(first_class) x systemd(managed)`
**Target:** solace-browser at `/home/phuc/projects/solace-browser/`
**Floors probed:** F1 (.deb + AppImage), F2 (.desktop Entry), F3 (XDG Base Directories), F4 (systemd Service), F5 (Wayland vs X11), F6 (Package Dependencies), F7 (File Permissions), F8 (D-Bus Integration), F9 (Secret Service / Keyring), F10 (CLI-First Integration)
**Protocol:** FORECAST -> CROSS-REF -> AUDIT -> FIX -> VERIFY -> SEAL -> POSTMORTEM
**Total Questions:** 343 (49 floors x 7 questions)
**Auth:** 65537

In [ ]:
# Cell 1: Bootstrap -- imports, paths, helper functions
# Tower 20: Linux QA -- probing solace-browser for Linux platform readiness
import os, sys, re, json, hashlib, glob as globmod
from pathlib import Path
from datetime import datetime

NORTHSTAR = "linux-qa"
NOTEBOOK_PATH = "notebooks/qa/11-linux-qa.ipynb"
TOWER = 20
TOWER_NAME = "Tower of Linux"
MIN_SCORE = 30  # early-stage Linux packaging

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
assert BROWSER_ROOT.exists(), f"FATAL: BROWSER_ROOT {BROWSER_ROOT} does not exist"

WEB = BROWSER_ROOT / "web"
SRC = BROWSER_ROOT / "src"
SCRIPTS_SRC = SRC / "scripts"
SCRIPTS_TOP = BROWSER_ROOT / "scripts"
DATA = BROWSER_ROOT / "data" / "default"
CSS_FILE = WEB / "css" / "site.css"
JS_DIR = WEB / "js"

results = []

def record(probe_id, name, passed, detail=""):
    """Record a PASS/FAIL result with evidence detail."""
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    """Safely read a file, returning empty string if missing."""
    p = Path(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""

def file_exists(path):
    return Path(path).exists()

def grep_files(root, pattern, extensions=None):
    """Search files under root for regex pattern. Returns {relpath: [matching_lines]}."""
    if extensions is None:
        extensions = [".py", ".js", ".sh", ".json", ".toml", ".spec",
                      ".html", ".css", ".yml", ".yaml", ".md", ".cfg",
                      ".desktop", ".service", ".conf", ".nix"]
    hits = {}
    for ext in extensions:
        for fpath in Path(root).rglob(f"*{ext}"):
            if any(skip in fpath.parts for skip in (".git", "node_modules", ".venv", "dist")):
                continue
            try:
                text = fpath.read_text(errors="ignore")
                for i, line in enumerate(text.splitlines(), 1):
                    if re.search(pattern, line, re.IGNORECASE):
                        rel = str(fpath.relative_to(BROWSER_ROOT))
                        hits.setdefault(rel, []).append(f"L{i}: {line.strip()[:120]}")
            except Exception:
                pass
    return hits

# Read build scripts for reuse
build_linux_src = read_file(SCRIPTS_SRC / "build-linux.sh")
build_linux_top = read_file(SCRIPTS_TOP / "build-linux.sh")
combined_build = build_linux_src + build_linux_top

print(f"Bootstrap OK -- BROWSER_ROOT: {BROWSER_ROOT}")
print(f"Timestamp: {datetime.now().isoformat()}")

In [ ]:
# ============================================================
# Floor 1: .deb and .AppImage Packaging
# Persona: AppImage Expert + Debian/Ubuntu Packager (Tower 20, Floors 1-2)
# Questions probed:
#   Q1: Does the build script produce a .deb package?
#   Q2: Does the build script produce an .AppImage?
#   Q3: Is Tauri used for Linux packaging (produces both)?
#   Q4: Are SHA-256 checksums generated for Linux packages?
# ============================================================
print("=" * 60)
print("FLOOR 1: .deb and .AppImage Packaging")
print("=" * 60)

# Q1: .deb references
has_deb = bool(re.search(r"\.deb|dpkg|debian", combined_build, re.IGNORECASE))
record("F1-Q1", ".deb packaging in build scripts", has_deb,
       ".deb packaging found" if has_deb else "No .deb references")

# Q2: .AppImage references
has_appimage = bool(re.search(r"AppImage|appimage", combined_build, re.IGNORECASE))
record("F1-Q2", ".AppImage in build scripts", has_appimage,
       "AppImage packaging found" if has_appimage else "No AppImage references")

# Q3: Tauri for Linux packaging
has_tauri = bool(re.search(r"tauri|cargo|src-tauri", combined_build, re.IGNORECASE))
record("F1-Q3", "Tauri used for Linux packaging", has_tauri,
       "Tauri build pipeline found" if has_tauri else "No Tauri references")

# Q4: SHA-256 checksums
has_checksum = bool(re.search(r"sha256sum|shasum", combined_build))
record("F1-Q4", "SHA-256 checksums for packages", has_checksum,
       "Checksum generation found" if has_checksum else "No checksum generation")

In [ ]:
# ============================================================
# Floor 2: Desktop File (.desktop Entry)
# Persona: XDG Standards Expert (Tower 20, Floor 12)
# Questions probed:
#   Q1: Does a .desktop file exist in the project?
#   Q2: Does the build script create desktop entries?
#   Q3: Are icon files provided for the Linux desktop?
#   Q4: Does the .desktop file have correct [Desktop Entry] structure?
# ============================================================
print("=" * 60)
print("FLOOR 2: Desktop File (.desktop Entry)")
print("=" * 60)

# Q1: .desktop file exists
desktop_files = [f for f in BROWSER_ROOT.rglob("*.desktop")
                 if not any(s in str(f) for s in (".git", ".venv", "node_modules"))]
record("F2-Q1", ".desktop file exists", len(desktop_files) > 0,
       f"{len(desktop_files)} files: {[str(f.relative_to(BROWSER_ROOT)) for f in desktop_files[:3]]}")

# Q2: Build script creates desktop entries
desktop_build = grep_files(BROWSER_ROOT, r"\.desktop|desktop.entry|desktop.file|xdg-desktop",
                           extensions=[".sh", ".py", ".json"])
record("F2-Q2", "Desktop entry in build pipeline", len(desktop_build) > 0,
       f"{len(desktop_build)} files reference .desktop creation")

# Q3: Linux icon files (PNG, SVG in correct sizes)
icon_dirs = grep_files(BROWSER_ROOT, r"icons/.*x\d+|share/icons|hicolor",
                       extensions=[".sh", ".py", ".json", ".desktop"])
png_icons = [f for f in BROWSER_ROOT.rglob("*.png")
             if "icon" in str(f).lower()
             and not any(s in str(f) for s in (".git", ".venv", "node_modules"))]
record("F2-Q3", "Linux desktop icons provided",
       len(icon_dirs) > 0 or len(png_icons) > 0,
       f"Icon dir refs: {len(icon_dirs)}, PNG icon files: {len(png_icons)}")

# Q4: .desktop file structure
if desktop_files:
    dt_content = desktop_files[0].read_text(errors="ignore")
    has_entry = bool(re.search(r"\[Desktop Entry\]", dt_content))
    has_exec = bool(re.search(r"Exec=", dt_content))
    has_name = bool(re.search(r"Name=", dt_content))
    has_icon = bool(re.search(r"Icon=", dt_content))
    record("F2-Q4", ".desktop file has correct structure",
           has_entry and has_exec and has_name,
           f"[Desktop Entry]: {has_entry}, Exec: {has_exec}, Name: {has_name}, Icon: {has_icon}")
else:
    record("F2-Q4", ".desktop file has correct structure", False,
           "No .desktop file to check")

In [ ]:
# ============================================================
# Floor 3: XDG Base Directory Compliance
# Persona: XDG Standards Expert (Tower 20, Floor 12)
# Questions probed:
#   Q1: Does code use XDG_CONFIG_HOME, XDG_DATA_HOME, XDG_CACHE_HOME?
#   Q2: Does code use ~/.config/, ~/.local/share/, ~/.cache/ paths?
#   Q3: Does xdg-open get used for opening URLs or files?
#   Q4: Does code respect XDG_RUNTIME_DIR for temp data?
# ============================================================
print("=" * 60)
print("FLOOR 3: XDG Base Directory Compliance")
print("=" * 60)

# Q1: XDG env vars
xdg_env = grep_files(BROWSER_ROOT,
    r"XDG_CONFIG_HOME|XDG_DATA_HOME|XDG_CACHE_HOME",
    extensions=[".py", ".js", ".sh"])
record("F3-Q1", "XDG env vars (CONFIG/DATA/CACHE)", len(xdg_env) > 0,
       f"{len(xdg_env)} files" if xdg_env
       else "No XDG_CONFIG_HOME/XDG_DATA_HOME/XDG_CACHE_HOME usage")

# Q2: Conventional XDG paths
xdg_paths = grep_files(BROWSER_ROOT,
    r"\.config/|\.local/share/|\.cache/",
    extensions=[".py", ".js", ".sh"])
record("F3-Q2", "Conventional XDG paths (~/.config etc.)", len(xdg_paths) > 0,
       f"{len(xdg_paths)} files reference XDG conventional paths")

# Q3: xdg-open
xdg_open = grep_files(BROWSER_ROOT, r"xdg-open|xdg.open",
                      extensions=[".py", ".js", ".sh"])
record("F3-Q3", "xdg-open used for URLs/files", len(xdg_open) > 0,
       f"{len(xdg_open)} files" if xdg_open
       else "No xdg-open usage found")

# Q4: XDG_RUNTIME_DIR
xdg_runtime = grep_files(BROWSER_ROOT, r"XDG_RUNTIME_DIR",
                         extensions=[".py", ".js", ".sh"])
record("F3-Q4", "XDG_RUNTIME_DIR for temp data", len(xdg_runtime) > 0,
       f"{len(xdg_runtime)} files" if xdg_runtime
       else "No XDG_RUNTIME_DIR usage")

In [ ]:
# ============================================================
# Floor 4: systemd Service / Autostart
# Persona: systemd Integration Expert (Tower 20, Floor 8)
# Questions probed:
#   Q1: Is there a systemd .service file in the project?
#   Q2: Does code reference systemctl or journalctl?
#   Q3: Does the service file have correct [Unit]/[Service]/[Install]?
#   Q4: Are resource limits (MemoryLimit, CPUQuota) configured?
# ============================================================
print("=" * 60)
print("FLOOR 4: systemd Service / Autostart")
print("=" * 60)

# Q1: .service files
service_files = [f for f in BROWSER_ROOT.rglob("*.service")
                 if not any(s in str(f) for s in (".git", ".venv", "node_modules"))]
timer_files = [f for f in BROWSER_ROOT.rglob("*.timer")
               if not any(s in str(f) for s in (".git", ".venv", "node_modules"))]
record("F4-Q1", "systemd .service/.timer files",
       len(service_files) > 0 or len(timer_files) > 0,
       f"{len(service_files)} .service, {len(timer_files)} .timer")

# Q2: systemctl/journalctl references
systemd_refs = grep_files(BROWSER_ROOT,
    r"systemctl|journalctl|systemd",
    extensions=[".py", ".sh", ".md"])
record("F4-Q2", "systemctl/journalctl references", len(systemd_refs) > 0,
       f"{len(systemd_refs)} files")

# Q3: Service file structure check
if service_files:
    svc = service_files[0].read_text(errors="ignore")
    has_unit = bool(re.search(r"\[Unit\]", svc))
    has_svc = bool(re.search(r"\[Service\]", svc))
    has_install = bool(re.search(r"\[Install\]", svc))
    record("F4-Q3", "Service file structure",
           has_unit and has_svc and has_install,
           f"[Unit]: {has_unit}, [Service]: {has_svc}, [Install]: {has_install}")
else:
    # Check if systemd content exists inline in any file
    inline_svc = grep_files(BROWSER_ROOT, r"\[Service\]|ExecStart=|WantedBy=",
                            extensions=[".py", ".sh", ".conf"])
    record("F4-Q3", "Service file structure", len(inline_svc) > 0,
           f"Inline service config in {len(inline_svc)} files" if inline_svc
           else "No .service files or inline service config")

# Q4: Resource limits
resource_refs = grep_files(BROWSER_ROOT,
    r"MemoryLimit|CPUQuota|LimitNOFILE|MemoryMax|MemoryHigh",
    extensions=[".service", ".py", ".sh", ".conf"])
record("F4-Q4", "Resource limits configured", len(resource_refs) > 0,
       f"{len(resource_refs)} files" if resource_refs
       else "No systemd resource limits configured")

In [ ]:
# ============================================================
# Floor 5: Wayland vs X11 Support
# Persona: Wayland + X11 Testers (Tower 20, Floors 4-5)
# Questions probed:
#   Q1: Is there Wayland support (ozone-platform, GDK_BACKEND)?
#   Q2: Is there X11 support (DISPLAY, xorg references)?
#   Q3: Does code handle xdg-desktop-portal for Wayland screencap?
#   Q4: Does code handle X11 clipboard (CLIPBOARD vs PRIMARY)?
# ============================================================
print("=" * 60)
print("FLOOR 5: Wayland vs X11 Support")
print("=" * 60)

# Q1: Wayland
wayland_hits = grep_files(BROWSER_ROOT,
    r"wayland|ozone-platform|GDK_BACKEND.*wayland|WAYLAND_DISPLAY|pipewire",
    extensions=[".py", ".js", ".sh", ".json", ".conf"])
record("F5-Q1", "Wayland support references", len(wayland_hits) > 0,
       f"{len(wayland_hits)} files" if wayland_hits
       else "No Wayland references (ozone-platform, GDK_BACKEND)")

# Q2: X11
x11_hits = grep_files(BROWSER_ROOT,
    r"\bx11\b|\bDISPLAY\b|xorg|XAUTHORITY|xdotool|xclip|xsel",
    extensions=[".py", ".js", ".sh"])
record("F5-Q2", "X11 support references", len(x11_hits) > 0,
       f"{len(x11_hits)} files" if x11_hits
       else "No X11 / DISPLAY / xorg references")

# Q3: xdg-desktop-portal
portal_hits = grep_files(BROWSER_ROOT,
    r"xdg-desktop-portal|screen.?cast|org\.freedesktop\.portal",
    extensions=[".py", ".js", ".sh"])
record("F5-Q3", "xdg-desktop-portal for screencap", len(portal_hits) > 0,
       f"{len(portal_hits)} files" if portal_hits
       else "No xdg-desktop-portal references")

# Q4: X11 clipboard handling
clipboard_hits = grep_files(BROWSER_ROOT,
    r"CLIPBOARD|PRIMARY|xclip|xsel|wl-copy|wl-paste",
    extensions=[".py", ".js", ".sh"])
record("F5-Q4", "Clipboard handling (X11/Wayland)", len(clipboard_hits) > 0,
       f"{len(clipboard_hits)} files" if clipboard_hits
       else "No clipboard-specific code")

In [ ]:
# ============================================================
# Floor 6: Package Dependency Management
# Persona: Debian/Ubuntu Packager + Flatpak Expert (Tower 20, Floors 2-3)
# Questions probed:
#   Q1: Does the build script install Linux dependencies (apt-get)?
#   Q2: Is there a requirements.txt or pyproject.toml for Python deps?
#   Q3: Does the build reference Flatpak shared runtime?
#   Q4: Are native library dependencies documented?
# ============================================================
print("=" * 60)
print("FLOOR 6: Package Dependency Management")
print("=" * 60)

# Q1: apt-get / dnf / pacman in build
apt_hits = grep_files(BROWSER_ROOT,
    r"apt-get install|apt install|dnf install|pacman -S|zypper",
    extensions=[".sh", ".md", ".yml"])
record("F6-Q1", "System package install in build", len(apt_hits) > 0,
       f"{len(apt_hits)} files" if apt_hits
       else "No apt-get/dnf/pacman install commands")

# Q2: Python dependency files
req_exists = file_exists(BROWSER_ROOT / "requirements.txt")
pyproject_exists = file_exists(BROWSER_ROOT / "pyproject.toml")
setup_exists = file_exists(BROWSER_ROOT / "setup.py") or file_exists(BROWSER_ROOT / "setup.cfg")
record("F6-Q2", "Python dependency files exist",
       req_exists or pyproject_exists or setup_exists,
       f"requirements.txt: {req_exists}, pyproject.toml: {pyproject_exists}, setup.*: {setup_exists}")

# Q3: Flatpak shared runtime
flatpak_hits = grep_files(BROWSER_ROOT,
    r"flatpak|flathub|finish-args|shared.runtime",
    extensions=[".json", ".yaml", ".yml", ".sh"])
record("F6-Q3", "Flatpak shared runtime references", len(flatpak_hits) > 0,
       f"{len(flatpak_hits)} files" if flatpak_hits
       else "No Flatpak references")

# Q4: Native library docs (libwebkit, GTK, etc.)
lib_docs = grep_files(BROWSER_ROOT,
    r"libwebkit|libgtk|libayatana|librsvg|patchelf|webkit2gtk",
    extensions=[".sh", ".md", ".txt", ".yml"])
record("F6-Q4", "Native library dependencies documented", len(lib_docs) > 0,
       f"{len(lib_docs)} files reference native libs")

In [ ]:
# ============================================================
# Floor 7: File Permission Handling
# Persona: Filesystem Expert + Security Expert (Tower 20, Floors 37, 39)
# Questions probed:
#   Q1: Does code set appropriate file permissions (chmod, umask, mode)?
#   Q2: Does the AppImage get chmod +x after creation?
#   Q3: Does vault/credential storage set restricted permissions?
#   Q4: Are SUID/SGID bits avoided? Does the app avoid running as root?
# ============================================================
print("=" * 60)
print("FLOOR 7: File Permission Handling")
print("=" * 60)

# Q1: chmod / umask / mode
perm_hits = grep_files(BROWSER_ROOT,
    r"chmod|umask|os\.chmod|stat\.S_|0o[0-7]{3}|file.?mode|permission",
    extensions=[".py", ".sh"])
record("F7-Q1", "File permission handling (chmod/umask)", len(perm_hits) > 0,
       f"{len(perm_hits)} files set permissions" if perm_hits
       else "No chmod/umask/mode references")

# Q2: AppImage chmod +x
has_chmod_x = bool(re.search(r"chmod\s+\+x.*AppImage", combined_build))
record("F7-Q2", "AppImage gets chmod +x", has_chmod_x,
       "chmod +x found for AppImage" if has_chmod_x
       else "No chmod +x for AppImage in build script")

# Q3: Vault restricted permissions
vault_perm = grep_files(BROWSER_ROOT,
    r"0o600|0600|S_IRUSR|S_IWUSR|owner.only|private.?file",
    extensions=[".py"])
record("F7-Q3", "Vault uses restricted permissions (0600)", len(vault_perm) > 0,
       f"{len(vault_perm)} files set restricted perms" if vault_perm
       else "No 0600 / owner-only permission patterns")

# Q4: No SUID/SGID, no root required
root_check = grep_files(BROWSER_ROOT,
    r"os\.getuid|os\.geteuid|SUID|SGID|setuid|setgid|run.as.root|sudo",
    extensions=[".py", ".sh"])
record("F7-Q4", "Root/SUID avoidance", len(root_check) > 0,
       f"{len(root_check)} files reference root/SUID checks" if root_check
       else "No explicit root/SUID checks (may be fine if app never needs root)")

In [ ]:
# ============================================================
# Floor 8: D-Bus Integration
# Persona: DBus Integration Expert (Tower 20, Floor 29)
# Questions probed:
#   Q1: Does code use D-Bus for IPC or notifications?
#   Q2: Is org.freedesktop.Notifications used?
#   Q3: Does the macOS spec exclude dbus (Linux-only module)?
#   Q4: Is there any GDBus or dbus-send usage?
# ============================================================
print("=" * 60)
print("FLOOR 8: D-Bus Integration")
print("=" * 60)

# Q1: D-Bus references
dbus_hits = grep_files(BROWSER_ROOT,
    r"\bdbus\b|d-bus|DBus|dbus-python|pydbus|jeepney|dbus-next",
    extensions=[".py", ".js", ".sh", ".json"])
record("F8-Q1", "D-Bus references in codebase", len(dbus_hits) > 0,
       f"{len(dbus_hits)} files" if dbus_hits
       else "No D-Bus references")

# Q2: org.freedesktop.Notifications
notif_dbus = grep_files(BROWSER_ROOT,
    r"org\.freedesktop\.Notifications|org\.freedesktop\.portal",
    extensions=[".py", ".js"])
record("F8-Q2", "freedesktop Notifications/portal", len(notif_dbus) > 0,
       f"{len(notif_dbus)} files" if notif_dbus
       else "No org.freedesktop.Notifications usage")

# Q3: dbus excluded in macOS spec (platform isolation)
macos_spec = read_file(BROWSER_ROOT / "solace-browser-macos.spec")
dbus_excluded = bool(re.search(r"'dbus'", macos_spec))
record("F8-Q3", "dbus excluded in macOS spec", dbus_excluded,
       "dbus correctly excluded from macOS build" if dbus_excluded
       else "dbus not excluded in macOS spec (may cause import errors on macOS)")

# Q4: GDBus / dbus-send
gdbus_hits = grep_files(BROWSER_ROOT,
    r"GDBus|gdbus|dbus-send|dbus-monitor|busctl",
    extensions=[".py", ".sh"])
record("F8-Q4", "GDBus / dbus-send usage", len(gdbus_hits) > 0,
       f"{len(gdbus_hits)} files" if gdbus_hits
       else "No GDBus or dbus-send usage")

In [ ]:
# ============================================================
# Floor 9: Secret Service / Keyring
# Persona: Secret Service Expert (Tower 20, Floor 17)
# Questions probed:
#   Q1: Does code reference libsecret or GNOME Keyring?
#   Q2: Is there KDE Wallet (kwallet) integration?
#   Q3: Does the vault have AES-256-GCM encrypted fallback?
#   Q4: Is there PBKDF2 key derivation for vault encryption?
# ============================================================
print("=" * 60)
print("FLOOR 9: Secret Service / Keyring")
print("=" * 60)

# Q1: libsecret / GNOME Keyring
secret_hits = grep_files(BROWSER_ROOT,
    r"libsecret|gnome.keyring|SecretService|org\.freedesktop\.secrets|keytar",
    extensions=[".py", ".js", ".json"])
record("F9-Q1", "libsecret / GNOME Keyring", len(secret_hits) > 0,
       f"{len(secret_hits)} files" if secret_hits
       else "No libsecret/GNOME Keyring references")

# Q2: KDE Wallet
kwallet_hits = grep_files(BROWSER_ROOT,
    r"kwallet|kde.wallet|KWallet",
    extensions=[".py", ".js"])
record("F9-Q2", "KDE Wallet integration", len(kwallet_hits) > 0,
       f"{len(kwallet_hits)} files" if kwallet_hits
       else "No KDE Wallet references")

# Q3: AES-256-GCM
aes_hits = grep_files(BROWSER_ROOT,
    r"AES.*GCM|aes.256.gcm|AESGCM|Fernet|encrypt",
    extensions=[".py"])
record("F9-Q3", "AES-256-GCM encrypted vault", len(aes_hits) > 0,
       f"{len(aes_hits)} files" if aes_hits
       else "No AES-GCM encryption references")

# Q4: PBKDF2
pbkdf_hits = grep_files(BROWSER_ROOT,
    r"PBKDF2|pbkdf2|key.?deriv|scrypt|argon2",
    extensions=[".py"])
record("F9-Q4", "PBKDF2 key derivation", len(pbkdf_hits) > 0,
       f"{len(pbkdf_hits)} files" if pbkdf_hits
       else "No PBKDF2/scrypt/argon2 key derivation")

In [ ]:
# ============================================================
# Floor 10: CLI-First Integration
# Persona: CLI-First Linux User (Tower 20, Floor 11)
# Questions probed:
#   Q1: Are there shell scripts for Linux operations?
#   Q2: Do shell scripts have proper shebangs (#!/bin/bash)?
#   Q3: Does code handle SIGTERM/SIGINT/SIGHUP signals?
#   Q4: Is there a headless / no-GUI mode for servers/CI?
# ============================================================
print("=" * 60)
print("FLOOR 10: CLI-First Integration")
print("=" * 60)

# Q1: Shell scripts
sh_files = [f for f in BROWSER_ROOT.rglob("*.sh")
            if not any(s in str(f) for s in (".git", ".venv", "node_modules", "dist"))]
record("F10-Q1", "Shell scripts for Linux ops", len(sh_files) > 0,
       f"{len(sh_files)} .sh files")

# Q2: Proper shebangs
shebang_count = 0
for f in sh_files[:20]:  # sample first 20
    try:
        first_line = f.read_text(errors="ignore").split("\n")[0]
        if re.match(r"^#!/", first_line):
            shebang_count += 1
    except Exception:
        pass
sampled = min(len(sh_files), 20)
record("F10-Q2", "Shell scripts have shebangs",
       shebang_count > 0 and shebang_count >= sampled * 0.8,
       f"{shebang_count}/{sampled} sampled scripts have shebangs")

# Q3: Signal handling
signal_hits = grep_files(BROWSER_ROOT,
    r"SIGTERM|SIGINT|SIGHUP|signal\.signal|signal\.SIGTERM",
    extensions=[".py"])
record("F10-Q3", "Signal handling (SIGTERM/SIGINT)", len(signal_hits) > 0,
       f"{len(signal_hits)} files handle signals" if signal_hits
       else "No SIGTERM/SIGINT/SIGHUP handling")

# Q4: Headless mode
headless_hits = grep_files(BROWSER_ROOT,
    r"headless|--no-gui|--headless|DISPLAY.*:99|xvfb",
    extensions=[".py", ".sh"])
record("F10-Q4", "Headless / no-GUI mode", len(headless_hits) > 0,
       f"{len(headless_hits)} files" if headless_hits
       else "No headless mode references")

In [ ]:
# ============================================================
# EVIDENCE SUMMARY -- Tower 20: Linux QA x Solace Browser
# ============================================================

passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
pass_rate = (passed / total * 100) if total > 0 else 0

floor_summary = {}
for r in results:
    floor = r["id"].split("-")[0]
    if floor not in floor_summary:
        floor_summary[floor] = {"passed": 0, "failed": 0, "total": 0}
    floor_summary[floor]["total"] += 1
    if r["status"] == "PASS":
        floor_summary[floor]["passed"] += 1
    else:
        floor_summary[floor]["failed"] += 1

floor_labels = {
    "F1": ".deb + AppImage", "F2": ".desktop Entry",
    "F3": "XDG Base Dirs", "F4": "systemd Service",
    "F5": "Wayland vs X11", "F6": "Package Dependencies",
    "F7": "File Permissions", "F8": "D-Bus Integration",
    "F9": "Secret Service", "F10": "CLI-First",
}

summary = {
    "surface": NORTHSTAR,
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total,
    "passed": passed,
    "failed": failed,
    "pass_rate_pct": round(pass_rate, 1),
    "verdict": "STRONG" if pass_rate >= 70 else "PARTIAL" if pass_rate >= 40 else "WEAK",
    "floors": {}
}

print(f"\n{'='*60}")
print(f"TOWER 20: Linux QA -- EVIDENCE SUMMARY")
print(f"{'='*60}")
print(f"Total probes: {total}")
print(f"PASS: {passed} ({pass_rate:.1f}%)")
print(f"FAIL: {failed}")
print(f"Verdict: {summary['verdict']}")
print(f"\n{'Floor':<6} {'Label':<25} {'Pass':<6} {'Fail':<6} {'Total':<6}")
print("-" * 55)
for fid in sorted(floor_summary.keys(), key=lambda x: int(x[1:])):
    fs = floor_summary[fid]
    label = floor_labels.get(fid, "Unknown")
    print(f"{fid:<6} {label:<25} {fs['passed']:<6} {fs['failed']:<6} {fs['total']:<6}")
    summary["floors"][fid] = {"label": label, **fs}

print(f"\n{'='*60}")
print("Evidence JSON:")
print(json.dumps(summary, indent=2))